**Setup:** download `data/zip/fintech.zip` from the DA Launchpad Datasets page and unzip it in the same folder as this notebook, so that paths like `fintech/<table>.csv` work. Needs pandas (and matplotlib/seaborn for charts).

# Cleaning UPI transactions

This notebook cleans the fintech transactions table. The raw file has rupee text in `amount`, two date formats in `txn_datetime`, and duplicate rows. You will fix each one and write a cleaning log.

Put the CSV files for this dataset in a folder named `fintech` next to this notebook. The site zip has one folder per dataset. Run the cells from top to bottom.

In [ ]:
import pandas as pd

df = pd.read_csv("fintech/transactions.csv")
print("raw rows:", df.shape[0])
df.head()

## Step 1: Look at the raw text

Check which values have rupee signs or commas, and which dates use the day-first format.

In [ ]:
print("amount has rupee sign or comma:", df["amount"].str.contains("₹|,").sum())
print("date starts with a 4-digit year:", df["txn_datetime"].str.match(r"^\d{4}-").sum())
print("date uses DD-MM-YYYY:", (~df["txn_datetime"].str.match(r"^\d{4}-")).sum())

## Step 2: Remove duplicate rows

The same `txn_id` can appear more than once. Keep the first row for each `txn_id`.

In [ ]:
exact_dupes = df.duplicated().sum()
id_dupes = df["txn_id"].duplicated().sum()
df = df.drop_duplicates(subset="txn_id", keep="first").copy()
print("exact duplicate rows:", exact_dupes)
print("duplicate txn_id rows removed:", id_dupes)
print("rows now:", len(df))

## Step 3: Turn the amount text into a number

Remove the rupee sign, the commas and any spaces. Then convert to float.

In [ ]:
df["amount_inr"] = df["amount"].str.replace(r"[₹,\s]", "", regex=True).astype(float)
print(df["amount_inr"].describe().round(2))

## Step 4: Parse both date formats

Try the ISO format first. Then try the day-first format for the rows that failed. `combine_first` keeps the first good value.

In [ ]:
iso = pd.to_datetime(df["txn_datetime"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
dmy = pd.to_datetime(df["txn_datetime"], format="%d-%m-%Y %H:%M:%S", errors="coerce")
df["txn_ts"] = iso.combine_first(dmy)
print("rows that could not be parsed:", df["txn_ts"].isna().sum())
print(df["txn_ts"].min(), "to", df["txn_ts"].max())

## Step 5: Look at the largest amounts

Do not delete outliers without a reason. Flag them first, and show the top values to a business owner. A 3 x IQR fence is only a starting rule: many flagged rows are real large payments, so the flag is for review, not removal.

In [ ]:
print(df.nlargest(5, "amount_inr")[["txn_id", "amount_inr", "payment_app", "txn_type"]])
q1, q3 = df["amount_inr"].quantile([0.25, 0.75])
upper = q3 + 3 * (q3 - q1)
df["amount_flag"] = df["amount_inr"] > upper
print("upper fence (Q3 + 3 x IQR):", round(upper, 2))
print("flagged rows:", int(df["amount_flag"].sum()))

## Step 6: Save the cleaned table and a cleaning log

The log lists every change, so a reviewer can check the numbers. The flag stays in the file, so no row is lost.

In [ ]:
clean = df[["txn_id", "user_id", "txn_type", "txn_ts", "amount_inr", "payment_app", "bank", "status", "amount_flag"]].copy()
clean.to_csv("transactions_clean.csv", index=False)

log = pd.DataFrame([
    {"step": "remove duplicate txn_id", "rows_removed": int(id_dupes)},
    {"step": "amount text to number", "rows_changed": int(df["amount"].str.contains("₹|,").sum())},
    {"step": "parse two date formats", "rows_unparsed": int(df["txn_ts"].isna().sum())},
    {"step": "flag amounts above upper fence", "rows_flagged": int(df["amount_flag"].sum())},
])
log.to_csv("cleaning_log.csv", index=False)
print(log.to_string(index=False))
print("saved rows:", len(clean))

## Findings to check

- Did any row lose its `txn_id` after cleaning? (It should not.)
- Is the cleaned row count the raw count minus the duplicate count?
- Which bank or app has the most flagged amounts? Write that down before you go on.